---
title : "Configuratie Agent"
execute:
    output: true
format:
    html:
        warning: false
        error: false
        code-fold: false
        css: style.css
        panel: center
---

De configuratie Agent van 'Talk with FEWS' heeft verschillende tools tot zijn beschikking.
Het taalmodel kan deze zelf invullen, maar hier wordt per tool een voorbeeld gegeven van de in en output. 

### Voorbereidingen
In eerste instantie worden de juiste variable opgehaald. Het context object bevat informatie over welke configuratie gebruikt wordt. In de voorbereiding is de configuratie al ingelezen in een database object. Dit is anders dan de normale 'FEWS database' waar alle data in zit. In deze notebook is alles met 'database' dus de configuratie database.


In [1]:
from pydantic_ai import format_as_xml
from setup_quarto import setup_dot_env_docs

ctx = setup_dot_env_docs()

## Search workflows{#sec-search-workflows}
Om meer inzicht te krijgen in de configuratie van FEWS kan naar verschillende dingen gezocht worden.

De drie zoek mogelijkheden zijn:

- [`search_workflows`](#sec-search-workflows)
- [`search_timeseries`](#sec-search-timeseries)
- [`search_transformations`](#sec-search-transformations)

Workflows is een logisch start punt om de verschillende tools uit te leggen. 

Gegeven een (deel) van de workflow naam kan de informatie achter deze workflow worden opgehaald.

Bijvoorbeeld: een workflow die `im.wiwb.knmi` bevat, laten we zoek of er workflows zijn die deze naam bevatten.

In [2]:
name_pattern = "knmi"

In [3]:
from talk_with_fews.tools.config.workflow_search import search_workflows
from talk_with_fews.models import (
    WorkflowSearchOutput,
    WorkflowSearchInput,
)

workflow_search_input = WorkflowSearchInput(name_pattern=name_pattern)
workflow_search_output: WorkflowSearchOutput = await search_workflows(
    ctx=ctx, input=workflow_search_input
)

In [4]:
print(format_as_xml(workflow_search_output, root_tag="WorkflowDetailsOutput"))

<WorkflowDetailsOutput>
  <results>
    <example>
      <id>27</id>
      <wf_id>wf_AggregateSynop</wf_id>
      <desc>Bereken gemiddelde referentiegewasverdamping en wind voor heel HH Delfland en per afwateringseenheid uit de KNMI Synop data</desc>
      <module_config_ids>
        <example>45</example>
        <example>46</example>
        <example>47</example>
        <example>48</example>
      </module_config_ids>
    </example>
    <example>
      <id>28</id>
      <wf_id>wf_ImportKnmiPostPro</wf_id>
      <desc>Bereken ensemble statistiek uit KNMI EPS data</desc>
      <module_config_ids>
        <example>49</example>
      </module_config_ids>
    </example>
  </results>
  <markdown_content>## Workflow Search Results
Found 2 workflows matching the pattern 'knmi'.

| ID | Workflow ID | Description | Module Config IDs |
|----|-------------|-------------|-------------------|
| 27 | wf_AggregateSynop | Bereken gemiddelde referentiegewasverdamping en wind voor heel HH Delfland en pe

## Get workflow details
Gegeven een workflow ID (bijvoorbeeld uit `search_workflows`) kan de bijbehorende informatie over deze workflow worden opgehaald.

In [5]:
workflow_id = workflow_search_output.results[0].id
workflow_id

27

In [6]:
from talk_with_fews.tools.config.workflows_details import get_workflow_details
from talk_with_fews.models import (
    WorkflowDetailsOutput,
    WorkflowDetailsInput,
)

workflow_details_input = WorkflowDetailsInput(workflow_id=workflow_id)
workflow_details_output: WorkflowDetailsOutput = await get_workflow_details(
    ctx=ctx, input=workflow_details_input
)

In [7]:
print(format_as_xml(workflow_details_output, root_tag="WorkflowDetailsOutput"))

<WorkflowDetailsOutput>
  <results>
    <example>
      <id>27</id>
      <wf_id>wf_AggregateSynop</wf_id>
      <description>Bereken gemiddelde referentiegewasverdamping en wind voor heel HH Delfland en per afwateringseenheid uit de KNMI Synop data</description>
      <wffn>wf_AggregateSynop</wffn>
      <module_calls>
        <example>
          <mic_id>45</mic_id>
          <modins>Aggregate_SynopToHHDelfland</modins>
          <ord_nr>0</ord_nr>
        </example>
        <example>
          <mic_id>46</mic_id>
          <modins>Aggregate_SynopToModel</modins>
          <ord_nr>0</ord_nr>
        </example>
        <example>
          <mic_id>47</mic_id>
          <modins>Aggregate_SynopToModel</modins>
          <ord_nr>0</ord_nr>
        </example>
        <example>
          <mic_id>48</mic_id>
          <modins>Aggregate_SynopToModel</modins>
          <ord_nr>0</ord_nr>
        </example>
      </module_calls>
    </example>
  </results>
  <markdown_content>## Workflow: wf_Agg

## Get transformation details{#sec-get-transformation-details}
Gegeven de transformatie ID kunnen we kijken wat verder met deze workflow te maken heeft. Bijvoorbeeld uit tabel in de `get_workflow_details` tool. 

In [8]:
transformation_id = 1357

In [9]:
from talk_with_fews.tools.config.transformation_details import (
    get_transformation_details,
)
from talk_with_fews.models import (
    TransformationDetailsInput,
    TransformationDetailsOutput,
)

transformation_details_input = TransformationDetailsInput(
    transformation_id=transformation_id
)
transformation_details_output: TransformationDetailsOutput = (
    await get_transformation_details(ctx=ctx, input=transformation_details_input)
)

In [10]:
print(
    format_as_xml(transformation_details_output, root_tag="TransformationDetailsOutput")
)

<TransformationDetailsOutput>
  <results>
    <example>
      <idx>1357</idx>
      <description />
      <ord_nr>3</ord_nr>
      <config_file>/ModuleConfigFiles/RTC_BOS2/RunRtcForecast.xml</config_file>
      <mic_id>167</mic_id>
      <modins>RunRtcForecast</modins>
      <time_series_links>
        <example>
          <inout>in</inout>
          <tss_id>692</tss_id>
        </example>
      </time_series_links>
    </example>
  </results>
  <markdown_content>## Transformation Details
**ID:** 1357
**Description:** 
**Config File:** /ModuleConfigFiles/RTC_BOS2/RunRtcForecast.xml

### Basic Information
| Property | Value |
|----------|--------|
| Order Number | 3 |
| Module Instance | RunRtcForecast |
| Module Config ID | 167 |

### Time Series Connections
This transformation processes the following time series:

#### Input Time Series
| ID | Description |
|----|-------------|
| 692 | Input time series |

#### Output Time Series
No output time series connected.

### Transformation Flo

## Describe Timeseries
Gegeven een timeseries ID kan je bijbehorende informatie over de transformaties ophalen waar deze timeseries in voor komt. 
Hier gebruiken we een timeseries ID uit [`get_transformation_details`](#sec-get-transformation-details). 

In [11]:
timeseries_id = transformation_details_output.results[0].time_series_links[0].tss_id
timeseries_id

692

In [12]:
from talk_with_fews.tools.config.timeseries_describe import describe_timeseries
from talk_with_fews.models import (
    DescribeTimeSeriesInput,
    DescribeTimeSeriesOutput,
)

timeseries_describe_input = DescribeTimeSeriesInput(timeseries_id=timeseries_id)
timeseries_describe_output: DescribeTimeSeriesOutput = await describe_timeseries(
    ctx=ctx, input=timeseries_describe_input
)

In [13]:
print(format_as_xml(timeseries_describe_output, root_tag="DescribeTimeSeriesOutput"))

<DescribeTimeSeriesOutput>
  <results>
    <example>
      <id>692</id>
      <modins>PreRtcHydroMasterNF</modins>
      <vtype>scalar</vtype>
      <parid>Q.setpoint</parid>
      <qualifiers>max/opgelegd</qualifiers>
      <tstype>simulated forecasting</tstype>
      <timess>5minute</timess>
      <description>PreRtcHydroMasterNF•scalar•Q.setpoint•max/opgelegd•simulated forecasting•5minute</description>
      <endpoint>False</endpoint>
      <endpoint_desc>null</endpoint_desc>
      <transformations>
        <example>
          <trafo_idx>1426</trafo_idx>
          <trafo_description />
          <ord_nr>28</ord_nr>
          <module_config_id>168</module_config_id>
          <module_config>RunRtcSimplePostprocessing</module_config>
          <workflow_id>50</workflow_id>
          <workflow_name>wf_RunRtcHydroMasterNF</workflow_name>
          <workflow_desc>Run RTC-tools Forecast (Hydromaster, no feedback)</workflow_desc>
          <inout>in</inout>
          <linked_tss_modins>Run

## Search Timeseries{#sec-search-timeseries}
Naast zoeken op workflows en dan via transformaties naar de bijbehorende timeseries, zoals met [`search_workflow`](#sec-search-workflows), kan je ook direct zoeken op timeseries.

Dit kan op basis van:
- parameter_id
- qualifier_pattern
- module_instance
- endpoint
- include_transformations

Hier worden bijbehorende timeseries opgehaald die voldoen aan de opgegeven criteria.

In [14]:
from talk_with_fews.tools.config.timeseries_search import search_timeseries
from talk_with_fews.models import (
    TimeSeriesSearchInput,
    TimeSeriesSearchOutput,
)

timeseries_search_input = TimeSeriesSearchInput(module_instance="im.wiwb.knmi")

timeseries_search_output: TimeSeriesSearchOutput = await search_timeseries(
    ctx=ctx, input=timeseries_search_input
)

In [15]:
print(format_as_xml(timeseries_search_output, root_tag="TimeSeriesSearchOutput"))

<TimeSeriesSearchOutput>
  <results />
  <markdown_content>## Time Series Search Results
Found 0 time series matching the criteria.

| ID | Module Instance | Type | Parameter | Qualifiers | Value Type | Time Step | Endpoint |
|----|----------------|------|-----------|------------|------------|-----------|----------|
</markdown_content>
</TimeSeriesSearchOutput>


## Search Transformation{#sec-search-transformations}
Als laatste van de searches: [`search_workflows`](#sec-search-workflows),[`search_workflows`](#sec-search-timeseries), kan je ook zoeken op transformaties. 
Dit is als configurator iets minder intuïtief, maar kan handig zijn voor de AI-agent om door te zoeken naar andere transformaties. 

Dit kan op basis van description_pattern, met als filter opties:
- workflow_name
- module_config
- max_depth

We gebruiken hier de zelfde name pattern als bij de workflow search, namelijk:

In [16]:
name_pattern

'knmi'

In [17]:
from talk_with_fews.tools.config.transformation_search import search_transformations
from talk_with_fews.models import (
    TransformationSearchInput,
    TransformationSearchOutput,
)

transformation_search_input = TransformationSearchInput(
    description_pattern=name_pattern
)
transformation_search_output: TransformationSearchOutput = await search_transformations(
    ctx=ctx, input=transformation_search_input
)

In [18]:
print(format_as_xml(transformation_search_output, root_tag="TimeSeriesSearchOutput"))

<TimeSeriesSearchOutput>
  <results />
  <markdown_content>## Transformation Search Results

Found 0 transformations matching the criteria.

No transformations found matching the criteria.
</markdown_content>
</TimeSeriesSearchOutput>


## Get database schema
De tools hier boven waren vooral gericht op het ophalen van informatie over de configuratie.
Vanaf hier worden specifiekere tools besproken die helpen bij het werken met de configuratie database.
Om op de juiste manier de tool execute query te gebruiken, is het handig om te weten wat er in de database zit.
Hiervoor zijn de schema's van de configuratie database.
Deze tool geeft een overzicht van de beschikbare tabellen en kolommen.

In [19]:
from talk_with_fews.tools.config.get_database_schema import get_database_schema
from talk_with_fews.models import (
    DatabaseSchemaOutput,
    DatabaseSchemaInput,
)

database_schema_input = DatabaseSchemaInput()

database_schema_output: DatabaseSchemaOutput = await get_database_schema(
    ctx=ctx, input=database_schema_input
)

In [20]:
print(format_as_xml(database_schema_output, root_tag="DatabaseSchemaOutput"))

<DatabaseSchemaOutput>
  <tables>
    <call_stacks>
      <example>id</example>
      <example>wf_id</example>
      <example>description</example>
      <example>wffn</example>
      <example>parent_id</example>
      <example>props_set_id</example>
    </call_stacks>
    <endpoints>
      <example>tss_id</example>
      <example>description</example>
    </endpoints>
    <links>
      <example>trafo_idx</example>
      <example>tss_id</example>
      <example>inout</example>
    </links>
    <modins_calls>
      <example>id</example>
      <example>call_stack_id</example>
      <example>modins</example>
      <example>props_set_id</example>
      <example>description</example>
    </modins_calls>
    <moduleinstances>
      <example>modin_id</example>
      <example>name</example>
    </moduleinstances>
    <moduleinstancesets>
      <example>set_id</example>
      <example>mid</example>
      <example>mip</example>
    </moduleinstancesets>
    <props_sets>
      <example>id</exampl

## Execute query
Deze tool geeft alle mogelijke vrijheid: het taalmodel kan elke SQL query uitvoeren.
Dit geeft het AI model alle vrijheid om de database te doorzoeken.

In [21]:
from talk_with_fews.tools.config.sql_query import execute_query
from talk_with_fews.models import SQLQueryInput, SQLQueryOutput

sql_query_input = SQLQueryInput(
    query="SELECT * FROM trafos",
)

sql_query_output: SQLQueryOutput = await execute_query(ctx=ctx, input=sql_query_input)

In [22]:
print(format_as_xml(sql_query_output, root_tag="SQLQueryOutput"))

<SQLQueryOutput>
  <results>
    <example>
      <idx>1</idx>
      <mic_id>27</mic_id>
      <ord_nr>1</ord_nr>
      <trafo_id>AggregatePvss.aggregate.Qdeb1.to.1h</trafo_id>
      <description />
      <config_file>/ModuleConfigFiles/Visualisatie/AggregatePvss.xml</config_file>
    </example>
    <example>
      <idx>2</idx>
      <mic_id>27</mic_id>
      <ord_nr>2</ord_nr>
      <trafo_id>AggregatePvss.aggregate.Qdeb2.to.1h</trafo_id>
      <description />
      <config_file>/ModuleConfigFiles/Visualisatie/AggregatePvss.xml</config_file>
    </example>
    <example>
      <idx>3</idx>
      <mic_id>27</mic_id>
      <ord_nr>3</ord_nr>
      <trafo_id>AggregatePvss.aggregate.Qdeb3.to.1h</trafo_id>
      <description />
      <config_file>/ModuleConfigFiles/Visualisatie/AggregatePvss.xml</config_file>
    </example>
    <example>
      <idx>4</idx>
      <mic_id>27</mic_id>
      <ord_nr>4</ord_nr>
      <trafo_id>AggregatePvss.cluster.gemiddelde</trafo_id>
      <description />
    

## Find Workflow Call Hierarchy

Om de workflows te begrijpen is het handig om te weten welke workflows aan elkaar gerelateerd zijn.
Deze tool geeft een overzicht van de workflow hiërarchie, oftewel welke workflows aan elkaar gerelateerd zijn.
Dit kan lang door gaan, daarom is er een `max_depth` parameter die aangeeft hoe diep de hiërarchie moet worden opgehaald.

In [23]:
root_workflow_id = workflow_search_output.results[0].id
root_workflow_id

27

In [24]:
from talk_with_fews.tools.config.workflow_hierarchy import find_workflow_call_hierarchy
from talk_with_fews.models import (
    WorkflowCallHierarchyInput,
)
from talk_with_fews.models import SQLQueryOutput as sql_query_output

workflow_call_hierarchy_input = WorkflowCallHierarchyInput(
    root_workflow_id=root_workflow_id
)
workflow_call_hierarchy_output: sql_query_output = await find_workflow_call_hierarchy(
    ctx=ctx, input=workflow_call_hierarchy_input
)

In [25]:
print(
    format_as_xml(
        workflow_call_hierarchy_output, root_tag="WorkflowCallHierarchyOutput"
    )
)

<WorkflowCallHierarchyOutput>
  <results>
    <example>
      <id>27</id>
      <wf_id>wf_AggregateSynop</wf_id>
      <description>Bereken gemiddelde referentiegewasverdamping en wind voor heel HH Delfland en per afwateringseenheid uit de KNMI Synop data</description>
      <parent_id>null</parent_id>
      <level>0</level>
    </example>
  </results>
  <column_names>
    <example>id</example>
    <example>wf_id</example>
    <example>description</example>
    <example>parent_id</example>
    <example>level</example>
  </column_names>
  <row_count>1</row_count>
  <query_executed>
    WITH RECURSIVE workflow_hierarchy AS (
        SELECT id, wf_id, description, parent_id, 0 AS level
        FROM call_stacks
        WHERE id = 27

        UNION ALL

        SELECT cs.id, cs.wf_id, cs.description, cs.parent_id, wh.level + 1
        FROM call_stacks cs
        JOIN workflow_hierarchy wh ON cs.parent_id = wh.id
    )
    SELECT id, wf_id, description, parent_id, level
    FROM workflow_hie

## Analyze Dependencies

Deze tool geeft een overzicht van de afhankelijkheden tussen de verschillende workflows.
Hierbij kun je aangeven welke richting van afhankelijkheden je wilt analyseren: upstream, downstream, of beide.
ook geef je aan welke entiteit je wilt analyseren: timeseries, transformation, of workflow.
Dit is eigenlijk een voorbereide versie van de `execute_query` tool, maar dan specifiek gericht op het analyseren van afhankelijkheden.

In [26]:
from talk_with_fews.tools.config.dependency_analysis import analyze_dependencies
from talk_with_fews.models import (
    DependencyAnalysisInput,
    DependencyAnalysisOutput,
)

analyze_dependencies_input = DependencyAnalysisInput(
    entity_type="transformation",
    entity_id=1,
    direction="downstream",  # 'upstream', 'downstream', or 'both'
)

analyze_dependencies_output: DependencyAnalysisOutput = await analyze_dependencies(
    ctx=ctx, input=analyze_dependencies_input
)

In [27]:
print(format_as_xml(analyze_dependencies_output, root_tag="DependencyAnalysisOutput"))

<DependencyAnalysisOutput>
  <results>
    <example>
      <tss_id>193</tss_id>
      <link_type>in</link_type>
    </example>
    <example>
      <tss_id>326</tss_id>
      <link_type>out</link_type>
    </example>
  </results>
  <markdown_content>## Dependencies for Transformation 1

### Input Time Series
| Time Series ID |
|----------------|
| 193 |

### Output Time Series
| Time Series ID |
|----------------|
| 326 |
</markdown_content>
</DependencyAnalysisOutput>


## Analyse Dead Points
Deze tool geeft een overzicht van de dode punten in de configuratie.
Dit is een uitgewerkte analyse tool die sp laat zien hoe we een speciefieke analyse kunnen uitvoeren op de configuratie database. 


In [28]:
from talk_with_fews.tools.config.dead_point_analysis import analyze_dead_points
from talk_with_fews.models import (
    DeadPointsAnalysisInput,
    DeadPointsAnalysisOutput,
)

dead_point_analysis_input = DeadPointsAnalysisInput(
    entity_type="transformation",
    entity_id=1,
)
dead_point_analysis_output: DeadPointsAnalysisOutput = await analyze_dead_points(
    ctx=ctx, input=dead_point_analysis_input
)

In [29]:
print(format_as_xml(dead_point_analysis_output, root_tag="DeadPointsAnalysisOutput"))

<DeadPointsAnalysisOutput>
  <count>103</count>
  <dead_points>
    <example>
      <id>331</id>
      <modins>AggregatePvss</modins>
      <vtype>scalar</vtype>
      <parid>H.meting</parid>
      <qualifiers>gemiddelde</qualifiers>
      <tstype>external historical</tstype>
      <timess>5minute</timess>
    </example>
    <example>
      <id>333</id>
      <modins>AggregatePvss</modins>
      <vtype>scalar</vtype>
      <parid>H.meting</parid>
      <qualifiers>afwijking</qualifiers>
      <tstype>external historical</tstype>
      <timess>5minute</timess>
    </example>
    <example>
      <id>349</id>
      <modins>Aggregate_HarmonieToHHDelfland</modins>
      <vtype>scalar</vtype>
      <parid>Eact.fc</parid>
      <qualifiers />
      <tstype>simulated forecasting</tstype>
      <timess>hour</timess>
    </example>
    <example>
      <id>357</id>
      <modins>Aggregate_HarmonieToHHDelfland</modins>
      <vtype>scalar</vtype>
      <parid>Wind.fc.dir</parid>
      <qualifiers 

## Visualize graph{#sec-visualize-graph}

Hier wordt een grafische weergave van de workflow hiërarchie gegeven.

In [30]:
from talk_with_fews.tools.config.graph_visualize import visualize_graph
from talk_with_fews.models import (
    GraphVisualizationInput,
    GraphVisualizationOutput,
)

graph_visualize_input = GraphVisualizationInput(
    root_entity_id=1, root_entity_type="workflow"
)

graph_visualize_output: GraphVisualizationOutput = await visualize_graph(
    ctx=ctx, input=graph_visualize_input
)

In [31]:
print(format_as_xml(graph_visualize_output, root_tag="DeadPointsAnalysisOutput"))

<DeadPointsAnalysisOutput>
  <nodes>
    <example>
      <id>1</id>
      <type>workflow</type>
      <label>wf_ImportMatroos</label>
      <properties>
        <description>Importeer Matroos data</description>
      </properties>
      <level>0</level>
    </example>
    <example>
      <id>1</id>
      <type>transformation</type>
      <label>AggregatePvss|</label>
      <properties>
        <order>1</order>
      </properties>
      <level>1</level>
    </example>
    <example>
      <id>193</id>
      <type>timeseries</type>
      <label>ImportPvss•scalar•Q.min.meting•deb1•external historical•5minute</label>
      <properties>
        <type>external historical</type>
        <timestep>5minute</timestep>
      </properties>
      <level>2</level>
    </example>
    <example>
      <id>1</id>
      <type>transformation</type>
      <label>AggregatePvss|</label>
      <properties>
        <order>1</order>
      </properties>
      <level>3</level>
    </example>
    <example>
      <i

De grafiek kunnen we vervolgens hier ook zien:

In [32]:
from IPython.display import Markdown, display

display(Markdown(graph_visualize_output.markdown_content))

```mermaid
graph LR
1["wf_ImportMatroos"]:::fill:#e8f5e9,stroke:#1b5e20
1["AggregatePvss|"]:::fill:#f3e5f5,stroke:#4a148c
193["ImportPvss•scalar•Q.min.meting•deb1•external historical•5minute"]:::fill:#e1f5fe,stroke:#01579b
1["AggregatePvss|"]:::fill:#f3e5f5,stroke:#4a148c
0["AggregatePvss|Q.min.meting|deb1"]:::fill:#e1f5fe,stroke:#01579b
439["ImportPvssPostPro|"]:::fill:#f3e5f5,stroke:#4a148c
0["ImportPvss|Q.min.meting|deb1"]:::fill:#e1f5fe,stroke:#01579b
576["MergeForScada|"]:::fill:#f3e5f5,stroke:#4a148c
0["RunRtcPostHydroMaster|Q.min.sim|"]:::fill:#e1f5fe,stroke:#01579b
576["MergeForScada|"]:::fill:#f3e5f5,stroke:#4a148c
0["MergeForScada|Q.min.sim|merged"]:::fill:#e1f5fe,stroke:#01579b
501["MergeForDisplays|"]:::fill:#f3e5f5,stroke:#4a148c
0["MergeForDisplaysHarmonie|Q.min.sim|"]:::fill:#e1f5fe,stroke:#01579b
501["MergeForDisplays|"]:::fill:#f3e5f5,stroke:#4a148c
0["PreRtcHarmonie|Q.min.sim|"]:::fill:#e1f5fe,stroke:#01579b
502["MergeForDisplays|"]:::fill:#f3e5f5,stroke:#4a148c
0["MergeForDisplaysHarmonie|Q.min.sim|"]:::fill:#e1f5fe,stroke:#01579b
502["MergeForDisplays|"]:::fill:#f3e5f5,stroke:#4a148c
0["PreRtcToolsHarmonie|Q.min.sim|"]:::fill:#e1f5fe,stroke:#01579b
876["PreRtcForecastMisc|"]:::fill:#f3e5f5,stroke:#4a148c
0["ImportPvss|Q.min.meting|deb2"]:::fill:#e1f5fe,stroke:#01579b
876["PreRtcForecastMisc|"]:::fill:#f3e5f5,stroke:#4a148c
0["PreRtcForecastMisc|Q.meting|"]:::fill:#e1f5fe,stroke:#01579b
514["MergeForDisplays|"]:::fill:#f3e5f5,stroke:#4a148c
0["MergeForDisplaysHydroMaster|Q.min.sim|"]:::fill:#e1f5fe,stroke:#01579b
514["MergeForDisplays|"]:::fill:#f3e5f5,stroke:#4a148c
0["PreRtcHydroMaster|Q.min.sim|"]:::fill:#e1f5fe,stroke:#01579b
515["MergeForDisplays|"]:::fill:#f3e5f5,stroke:#4a148c
0["MergeForDisplaysHydroMaster|Q.min.sim|"]:::fill:#e1f5fe,stroke:#01579b
515["MergeForDisplays|"]:::fill:#f3e5f5,stroke:#4a148c
0["PreRtcToolsHydroMaster|Q.min.sim|"]:::fill:#e1f5fe,stroke:#01579b
893["PreRtcForecastMisc|"]:::fill:#f3e5f5,stroke:#4a148c
0["ImportPvss|Q.min.meting|deb2"]:::fill:#e1f5fe,stroke:#01579b
893["PreRtcForecastMisc|"]:::fill:#f3e5f5,stroke:#4a148c
0["PreRtcForecastMisc|Q.meting|"]:::fill:#e1f5fe,stroke:#01579b
540["MergeForDisplays|"]:::fill:#f3e5f5,stroke:#4a148c
0["MergeForDisplaysHydroMasterNF|Q.min.sim|"]:::fill:#e1f5fe,stroke:#01579b
540["MergeForDisplays|"]:::fill:#f3e5f5,stroke:#4a148c
0["PreRtcHydroMasterNF|Q.min.sim|"]:::fill:#e1f5fe,stroke:#01579b
541["MergeForDisplays|"]:::fill:#f3e5f5,stroke:#4a148c
0["MergeForDisplaysHydroMasterNF|Q.min.sim|"]:::fill:#e1f5fe,stroke:#01579b
541["MergeForDisplays|"]:::fill:#f3e5f5,stroke:#4a148c
0["PreRtcToolsHydroMasterNF|Q.min.sim|"]:::fill:#e1f5fe,stroke:#01579b
927["PreRtcForecastMisc|"]:::fill:#f3e5f5,stroke:#4a148c
0["ImportPvss|Q.min.meting|deb2"]:::fill:#e1f5fe,stroke:#01579b
927["PreRtcForecastMisc|"]:::fill:#f3e5f5,stroke:#4a148c
0["PreRtcForecastMisc|Q.meting|"]:::fill:#e1f5fe,stroke:#01579b
527["MergeForDisplays|"]:::fill:#f3e5f5,stroke:#4a148c
0["MergeForDisplaysRainguru|Q.min.sim|"]:::fill:#e1f5fe,stroke:#01579b
527["MergeForDisplays|"]:::fill:#f3e5f5,stroke:#4a148c
193 -->|in AggregatePvss| 1:::stroke:#01579b
1 -->|out AggregatePvss| 0:::stroke:#1b5e20
193 -->|in ImportPvssPostPro| 439:::stroke:#01579b
439 -->|out ImportPvssPostPro| 0:::stroke:#1b5e20
193 -->|in MergeForScada| 576:::stroke:#01579b
576 -->|out MergeForScada| 0:::stroke:#1b5e20
193 -->|in MergeForScada| 576:::stroke:#01579b
576 -->|out MergeForScada| 0:::stroke:#1b5e20
193 -->|in MergeForDisplays| 501:::stroke:#01579b
501 -->|out MergeForDisplays| 0:::stroke:#1b5e20
193 -->|in MergeForDisplays| 501:::stroke:#01579b
501 -->|out MergeForDisplays| 0:::stroke:#1b5e20
193 -->|in MergeForDisplays| 502:::stroke:#01579b
502 -->|out MergeForDisplays| 0:::stroke:#1b5e20
193 -->|in MergeForDisplays| 502:::stroke:#01579b
502 -->|out MergeForDisplays| 0:::stroke:#1b5e20
193 -->|in PreRtcForecastMisc| 876:::stroke:#01579b
876 -->|out PreRtcForecastMisc| 0:::stroke:#1b5e20
193 -->|in PreRtcForecastMisc| 876:::stroke:#01579b
876 -->|out PreRtcForecastMisc| 0:::stroke:#1b5e20
193 -->|in MergeForDisplays| 514:::stroke:#01579b
514 -->|out MergeForDisplays| 0:::stroke:#1b5e20
193 -->|in MergeForDisplays| 514:::stroke:#01579b
514 -->|out MergeForDisplays| 0:::stroke:#1b5e20
193 -->|in MergeForDisplays| 515:::stroke:#01579b
515 -->|out MergeForDisplays| 0:::stroke:#1b5e20
193 -->|in MergeForDisplays| 515:::stroke:#01579b
515 -->|out MergeForDisplays| 0:::stroke:#1b5e20
193 -->|in PreRtcForecastMisc| 893:::stroke:#01579b
893 -->|out PreRtcForecastMisc| 0:::stroke:#1b5e20
193 -->|in PreRtcForecastMisc| 893:::stroke:#01579b
893 -->|out PreRtcForecastMisc| 0:::stroke:#1b5e20
193 -->|in MergeForDisplays| 540:::stroke:#01579b
540 -->|out MergeForDisplays| 0:::stroke:#1b5e20
193 -->|in MergeForDisplays| 540:::stroke:#01579b
540 -->|out MergeForDisplays| 0:::stroke:#1b5e20
193 -->|in MergeForDisplays| 541:::stroke:#01579b
541 -->|out MergeForDisplays| 0:::stroke:#1b5e20
193 -->|in MergeForDisplays| 541:::stroke:#01579b
541 -->|out MergeForDisplays| 0:::stroke:#1b5e20
193 -->|in PreRtcForecastMisc| 927:::stroke:#01579b
927 -->|out PreRtcForecastMisc| 0:::stroke:#1b5e20
193 -->|in PreRtcForecastMisc| 927:::stroke:#01579b
927 -->|out PreRtcForecastMisc| 0:::stroke:#1b5e20
193 -->|in MergeForDisplays| 527:::stroke:#01579b
527 -->|out MergeForDisplays| 0:::stroke:#1b5e20
193 -->|in MergeForDisplays| 527:::stroke:#01579b
193 -->|input AggregatePvss| 1:::stroke:#01579b
1 -->|in ImportMATROOS| 1:::stroke:#4a148c
```

In [33]:
from talk_with_fews.tools.config.parameter_search import search_parameters
from talk_with_fews.models import GetParameterInput, GetParameterOutput

get_parameter_input = GetParameterInput(number_of_parameters=10)

get_parameter_output: GetParameterOutput = await search_parameters(
    ctx, get_parameter_input
)

In [34]:
print(format_as_xml(get_parameter_output, root_tag="GetParameterOutput"))

<GetParameterOutput>
  <parameters>
    <example>
      <id>ALARM</id>
    </example>
    <example>
      <id>ALARM.ne</id>
    </example>
    <example>
      <id>BOS</id>
    </example>
    <example>
      <id>BOS.ne</id>
    </example>
    <example>
      <id>BOS.uit</id>
    </example>
    <example>
      <id>ChlorideConc.met</id>
    </example>
    <example>
      <id>ChlorideConc.sim</id>
    </example>
    <example>
      <id>DummyWqRvw</id>
    </example>
    <example>
      <id>E.ref.Makkink.fc</id>
    </example>
    <example>
      <id>E.ref.Makkink.merged</id>
    </example>
  </parameters>
  <description>Found 10 parameter IDs matching the criteria. These parameter IDs can be used to search for timeseries using the search_timeseries tool.</description>
</GetParameterOutput>
